<a href="https://colab.research.google.com/github/zk025815-beep/HCCDA-AI-Course/blob/main/MedQuery_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install pypdf chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.8 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Suc

In [2]:
import pandas as pd

# Curated Real-World Clinical FAQ Dataset (Modeled on NIH MedlinePlus & WHO Guidelines)
clinical_faqs = [
    {
        "id": "MED-001",
        "category": "Infectious Diseases",
        "condition": "Dengue Fever",
        "source": "WHO / NIH Clinical Guidelines",
        "question": "What are the warning signs of severe Dengue fever and which painkillers must be avoided?",
        "answer": "High fever (40°C/104°F) is accompanied by severe headache, pain behind the eyes, muscle and joint pains, and rash. WARNING SIGNS of Severe Dengue (occurring 3–7 days after illness onset) include severe abdominal pain, persistent vomiting, rapid breathing, bleeding gums or nose, fatigue, and blood in vomit or stool. CRITICAL MEDICATION RULE: Patients must take ONLY Paracetamol (Acetaminophen) for fever and pain. Strictly AVOID Aspirin, Ibuprofen, Naproxen, and other NSAIDs, as they thin the blood and significantly increase the risk of fatal internal bleeding (Dengue Hemorrhagic Fever)."
    },
    {
        "id": "MED-002",
        "category": "Infectious Diseases",
        "condition": "Dengue Fever",
        "source": "WHO / NIH Clinical Guidelines",
        "question": "How is Dengue managed at home and when is hospitalization required?",
        "answer": "Mild dengue is managed at home with strict bed rest, Oral Rehydration Salts (ORS), coconut water, fresh juices, and Paracetamol (maximum 4 grams per day for adults, spaced every 6 hours). Hospitalization is immediately required if platelet counts drop rapidly alongside warning signs, if hematocrit rises sharply (indicating plasma leakage), or if the patient cannot tolerate oral fluids."
    },
    {
        "id": "MED-003",
        "category": "Endocrinology & Metabolic",
        "condition": "Type 2 Diabetes Mellitus",
        "source": "NIH National Institute of Diabetes (NIDDK)",
        "question": "What are the diagnostic blood sugar thresholds for Prediabetes and Diabetes (HbA1c and Fasting)?",
        "answer": "According to standard ADA and NIH criteria: 1) Normal: Fasting Plasma Glucose (FPG) less than 100 mg/dL and HbA1c less than 5.7%. 2) Prediabetes: FPG between 100 mg/dL and 125 mg/dL, OR HbA1c between 5.7% and 6.4%. 3) Diabetes: FPG of 126 mg/dL or higher (after 8 hours of fasting), OR HbA1c of 6.5% or higher, OR a 2-hour Oral Glucose Tolerance Test (OGTT) level of 200 mg/dL or higher."
    },
    {
        "id": "MED-004",
        "category": "Endocrinology & Metabolic",
        "condition": "Hypoglycemia (Low Blood Sugar)",
        "source": "NIH National Institute of Diabetes (NIDDK)",
        "question": "How do you treat acute low blood sugar (Hypoglycemia) using the 15-15 Rule?",
        "answer": "Hypoglycemia occurs when blood glucose drops below 70 mg/dL. Symptoms include shakiness, sweating, dizziness, fast heartbeat, and confusion. Use the '15-15 Rule': Consume 15 grams of fast-acting carbohydrates (such as 4 ounces / half a cup of regular fruit juice, 1 tablespoon of sugar or honey, or 3–4 glucose tablets), wait 15 minutes, and recheck blood sugar. If it is still below 70 mg/dL, repeat another 15 grams. Do NOT give food or liquid if the person is unconscious; administer injectable or nasal Glucagon and call emergency services."
    },
    {
        "id": "MED-005",
        "category": "Cardiology",
        "condition": "Hypertension (High Blood Pressure)",
        "source": "American Heart Association / NIH NHLBI",
        "question": "What are the blood pressure categories and what constitutes a Hypertensive Crisis?",
        "answer": "Blood pressure categories for adults: 1) Normal: Less than 120/80 mmHg. 2) Elevated: Systolic 120–129 AND Diastolic less than 80 mmHg. 3) Hypertension Stage 1: Systolic 130–139 OR Diastolic 80–89 mmHg. 4) Hypertension Stage 2: Systolic 140 or higher OR Diastolic 90 or higher mmHg. 5) HYPERTENSIVE CRISIS (Medical Emergency): Blood pressure higher than 180/120 mmHg. If accompanied by chest pain, shortness of breath, back pain, numbness, or difficulty speaking, seek emergency hospital care immediately."
    },
    {
        "id": "MED-006",
        "category": "Gastroenterology",
        "condition": "Acute Gastroenteritis & Dehydration",
        "source": "WHO / CDC Clinical Guidance",
        "question": "When are antibiotics needed for acute diarrhea and how should ORS be prepared?",
        "answer": "Most cases of acute watery diarrhea are viral (such as Rotavirus or Norovirus) and do NOT require antibiotics. Inappropriate antibiotic use can worsen diarrhea and cause resistance. Antibiotics are only indicated for bloody diarrhea (dysentery), suspected Cholera with severe dehydration, or laboratory-confirmed Giardiasis. Primary treatment is Low-Osmolarity Oral Rehydration Salts (ORS) dissolved in 1 liter of clean boiled/cooled water, plus Zinc supplementation (20 mg daily for 10–14 days for children over 6 months)."
    },
    {
        "id": "MED-007",
        "category": "Respiratory Medicine",
        "condition": "Bronchial Asthma",
        "source": "NIH National Heart, Lung, and Blood Institute (NHLBI)",
        "question": "What is the difference between a Rescue Inhaler and a Controller Inhaler in Asthma?",
        "answer": "1) Rescue Inhalers (Short-Acting Beta Agonists like Salbutamol / Albuterol): Used for rapid relief during an acute asthma attack to relax airway muscles within minutes. They do not reduce long-term airway inflammation. 2) Controller Inhalers (Inhaled Corticosteroids like Budesonide or Fluticasone): Taken daily even when feeling well to prevent inflammation and mucus buildup. If a patient needs their rescue inhaler more than twice a week, their asthma is considered poorly controlled and requires physician review."
    },
    {
        "id": "MED-008",
        "category": "Emergency & Environmental",
        "condition": "Heat Exhaustion vs. Heatstroke",
        "source": "CDC / NIH Emergency Medicine",
        "question": "How do you differentiate between Heat Exhaustion and Heatstroke, and what is the first aid?",
        "answer": "Heat Exhaustion presents with heavy sweating, cold/pale/clammy skin, fast weak pulse, nausea, and dizziness; move the person to a cool room, loosen clothing, and give sips of water. HEATSTROKE is a life-threatening emergency: core body temperature rises above 103°F (39.4°C), skin becomes hot, red, dry or damp, pulse is fast and strong, accompanied by confusion or unconsciousness. First Aid for Heatstroke: Call emergency services immediately, cool the person rapidly with cold water immersion or ice packs on the neck, armpits, and groin, and do NOT give fluids to drink."
    }
]

df_med = pd.DataFrame(clinical_faqs)
df_med.to_csv("real_world_medical_faqs.csv", index=False)
print(f"✅ Loaded {len(df_med)} verified clinical FAQ records across {df_med['category'].nunique()} medical specialties!")
df_med[["id", "category", "condition", "question"]].head()

✅ Loaded 8 verified clinical FAQ records across 6 medical specialties!


,id,category,condition,question
0,MED-001,Infectious Diseases,Dengue Fever,What are the warning signs of severe Dengue fe...
1,MED-002,Infectious Diseases,Dengue Fever,How is Dengue managed at home and when is hosp...
2,MED-003,Endocrinology & Metabolic,Type 2 Diabetes Mellitus,What are the diagnostic blood sugar thresholds...
3,MED-004,Endocrinology & Metabolic,Hypoglycemia (Low Blood Sugar),How do you treat acute low blood sugar (Hypogl...
4,MED-005,Cardiology,Hypertension (High Blood Pressure),What are the blood pressure categories and wha...


In [12]:
import os
from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize the ChromaDB client and Embedding Model locally
chroma_client = chromadb.EphemeralClient()
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Reset and create our Medical ChromaDB Collection
try:
    chroma_client.delete_collection(name="medical_rag_kb")
except:
    pass

med_collection = chroma_client.create_collection(
    name="medical_rag_kb",
    metadata={"hnsw:space": "cosine"}
)

# Pure Python replacement for RecursiveCharacterTextSplitter to avoid langchain dependency issues
class SimpleTextSplitter:
    def __init__(self, chunk_size=650, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def split_text(self, text):
        chunks = []
        start = 0
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            chunks.append(text[start:end])
            if end == len(text):
                break
            start += (self.chunk_size - self.chunk_overlap)
        return chunks

med_splitter = SimpleTextSplitter(chunk_size=650, chunk_overlap=100)

def index_medical_data(extra_files=None):
    docs, metadatas, ids = [], [], []

    # 1. Index our Structured Clinical FAQ Dataset with Context Injection
    for _, row in df_med.iterrows():
        enriched_chunk = (
            f"CONDITION: {row['condition']} ({row['category']})\n"
            f"PATIENT QUESTION: {row['question']}\n"
            f"CLINICAL GUIDELINE ANSWER: {row['answer']}"
        )
        docs.append(enriched_chunk)
        metadatas.append({
            "id": row["id"],
            "category": row["category"],
            "condition": row["condition"],
            "source": row["source"]
        })
        ids.append(row["id"])

    # 2. Optional: Also allow uploading custom Medical PDFs (Lab reports, WHO PDFs, etc.)
    if extra_files:
        chunk_idx = 1
        for file_path in extra_files:
            fname = os.path.basename(file_path)
            if fname.lower().endswith(".pdf"):
                reader = PdfReader(file_path)
                for page_num, page in enumerate(reader.pages):
                    text = page.extract_text()
                    if not text or not text.strip():
                        continue
                    for ch in med_splitter.split_text(" ".join(text.split())):
                        docs.append(f"UPLOADED DOCUMENT ({fname}, Page {page_num+1}):\n{ch}")
                        metadatas.append({
                            "id": f"PDF-{chunk_idx}",
                            "category": "Uploaded Medical PDF",
                            "condition": fname,
                            "source": f"{fname} (Page {page_num+1})"
                        })
                        ids.append(f"PDF-{chunk_idx}")
                        chunk_idx += 1

    # Re-create collection cleanly and insert
    global med_collection
    try:
        chroma_client.delete_collection(name="medical_rag_kb")
    except:
        pass
    med_collection = chroma_client.create_collection(
        name="medical_rag_kb",
        metadata={"hnsw:space": "cosine"}
    )

    embeddings = embed_model.encode(docs).tolist()
    med_collection.add(ids=ids, documents=docs, embeddings=embeddings, metadatas=metadatas)
    return f"✅ Indexed {len(docs)} clinical knowledge chunks into ChromaDB!"

print(index_medical_data())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Indexed 8 clinical knowledge chunks into ChromaDB!


In [14]:
import os
from google.colab import userdata
from google import genai

# Load your Gemini API key from Colab Secrets (left sidebar Key icon)
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Define llm_client in memory
llm_client = genai.Client()
print("✅ llm_client is now defined and ready!")

✅ llm_client is now defined and ready!


In [24]:
from google.genai import types

MEDICAL_SYSTEM_PROMPT = """You are MedAssist AI, an Evidence-Based Clinical FAQ Assistant.
Your role is to answer patient and healthcare queries STRICTLY using the provided [CLINICAL CONTEXT].

STRICT MEDICAL SAFETY RULES:
1. GROUNDING ONLY: Use ONLY the facts, thresholds, and medications explicitly stated in the [CLINICAL CONTEXT]. Never invent drug dosages or treatments.
2. SOURCE CITATION: Cite the exact Record ID and Source Agency at the end of key facts, e.g., [MED-001 | WHO / NIH Clinical Guidelines].
3. RED-FLAG ALERTS: If the context mentions emergency warning signs (such as internal bleeding, BP > 180/120, unconsciousness, or core temperature > 103°F), highlight them clearly in bold as **🚨 Emergency Warning Signs**.
4. OUT-OF-SCOPE REFUSAL: If the answer is not in the [CLINICAL CONTEXT], state:
   "I do not have verified clinical guidelines for this question in my current medical knowledge base. Please consult a licensed physician."
5. MANDATORY DISCLAIMER: End every medical response with a one-line italicized disclaimer:
   "*Disclaimer: This information is from clinical reference guidelines for educational purposes and is not a substitute for professional medical diagnosis.*"
"""

def query_medical_rag(user_question: str, category_filter: str = "All Specialties", top_k: int = 3, max_distance: float = 0.70):
    query_vec = embed_model.encode([user_question]).tolist()

    # Apply ChromaDB Metadata Filtering if user selected a specific medical department
    where_clause = None if category_filter == "All Specialties" else {"category": category_filter}

    results = med_collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        where=where_clause,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    if not docs:
        return "⚠️ No records found under this medical specialty filter.", "No chunks retrieved."

    formatted_chunks = []
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
        header = f"[{meta['id']} | {meta['source']} | Specialty: {meta['category']} | Distance: {dist:.3f}]"
        formatted_chunks.append(f"{header}\n{doc}")

    inspector_output = ("\n" + "="*55 + "\n").join(formatted_chunks)

    # Hard Distance Guardrail (Blocks unrelated medical questions before calling the LLM)
    if dists[0] > max_distance:
        return (
            f"🛑 **Clinical Safety Guardrail Triggered:** Your question has a Cosine Distance of `{dists[0]:.3f}` "
            f"(exceeding the safety threshold of `{max_distance}`). I cannot provide medical information on topics "
            f"outside my verified clinical database. Please consult a healthcare professional.",
            inspector_output
        )

    prompt = f"""[CLINICAL CONTEXT]:
{inspector_output}

[PATIENT / USER QUESTION]:
{user_question}
"""
    response = llm_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=MEDICAL_SYSTEM_PROMPT,
            temperature=0.0
        )
    )
    return response.text, inspector_output

In [25]:
import gradio as gr
import traceback

specialties_list = ["All Specialties"] + sorted(df_med["category"].unique().tolist()) + ["Uploaded Medical PDF"]

def handle_med_chat(user_message, specialty, chat_history):
    if chat_history is None:
        chat_history = []
    if not user_message or not user_message.strip():
        return "", chat_history, "⚠️ Please enter a medical question."

    try:
        # This utilizes the Gemini-backed function defined in cell 8hW3YE8KyNaC
        reply, sources = query_medical_rag(user_message, category_filter=specialty)
    except Exception as e:
        reply = f"❌ **Backend Error:** `{str(e)}`"
        sources = traceback.format_exc()

    # ✅ Fix: Append dictionaries with 'role' and 'content' keys
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": reply})

    return "", chat_history, sources

def handle_pdf_upload(files):
    if not files:
        return "⚠️ Please select a PDF file first."
    paths = [f.name if hasattr(f, "name") else f for f in files]
    return index_medical_data(extra_files=paths)

with gr.Blocks() as med_demo:
    gr.Markdown("# 🏥 MedAssist AI — Clinical FAQ & Guideline RAG Assistant")
    gr.Markdown(
        "Powered by **Core NIH / WHO Clinical FAQs**, **ChromaDB Metadata Filtering**, and **Gemini 2.5 Flash** with strict non-hallucination guardrails."
    )

    with gr.Row():
        # Left Column: Chat + Specialty Filter
        with gr.Column(scale=3):
            specialty_dropdown = gr.Dropdown(
                choices=specialties_list,
                value="All Specialties",
                label="⚕ Filter by Medical Specialty (ChromaDB Metadata Filter)"
            )
            # Classic Chatbot widget configuration
            chatbot = gr.Chatbot(height=430, label="MedAssist Clinical Bot")
            msg_box = gr.Textbox(
                placeholder="e.g., Can I give Ibuprofen to a patient with high fever and suspected Dengue?",
                label="Ask a Medical / Clinical Question"
            )
            with gr.Row():
                submit_btn = gr.Button("Ask MedAssist", variant="primary")
                clear_btn = gr.Button("Clear Consultation")

            gr.Examples(
                examples=[
                    ["Can I take Ibuprofen or Aspirin for Dengue fever pain, and what are the warning signs?"],
                    ["My fasting blood sugar is 118 mg/dL and HbA1c is 6.1%. Do I have diabetes?"],
                    ["A diabetic patient's sugar dropped to 58 mg/dL and they are dizzy. What is the 15-15 rule?"],
                    ["What is the difference between a rescue inhaler and a controller inhaler for asthma?"],
                    ["Can you prescribe an antibiotic dosage for a severe migraine headache?"] # Guardrail test!
                ],
                inputs=msg_box,
                label="⌖ Click Any Clinical Test Case Below:"
            )

        # Right Column: Evidence Inspector & PDF Uploader
        with gr.Column(scale=2):
            gr.Markdown("###  Retrieved Clinical Evidence (ChromaDB)")
            source_inspector = gr.Textbox(
                label="Matched Guidelines, Source Agency & Cosine Distance",
                lines=14,
                interactive=False
            )
            gr.Markdown("###  Add Custom Medical PDFs (WHO Guides / Lab Manuals)")
            pdf_uploader = gr.File(label="Upload Medical PDF", file_count="multiple", file_types=[".pdf"])
            upload_btn = gr.Button("➕ Index Medical PDF into ChromaDB")
            upload_status = gr.Textbox(label="Database Status", value="✅ 8 Core NIH/WHO Clinical Guidelines Active", interactive=False)

    # Connect Events
    msg_box.submit(handle_med_chat, [msg_box, specialty_dropdown, chatbot], [msg_box, chatbot, source_inspector])
    submit_btn.click(handle_med_chat, [msg_box, specialty_dropdown, chatbot], [msg_box, chatbot, source_inspector])
    clear_btn.click(lambda: ("", [], ""), None, [msg_box, chatbot, source_inspector])
    upload_btn.click(handle_pdf_upload, inputs=[pdf_uploader], outputs=[upload_status])

med_demo.launch(share=True, debug=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6fcdec30b40c266927.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6fcdec30b40c266927.gradio.live
